# Look at model coefficients

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
import pandas as pd
import numpy as np

coef_dir = BASE_PATH / "perm_res/coef"

## Create tables

In [ ]:
def get_avg_coef_across_repeats(df):
    def get_summary(g):
        summary = {"avg_coef": np.mean(g["coef"]), "stdev_coef": np.std(g["coef"])}
        return pd.Series(summary)

    result = (
        df.groupby(
            [
                "repeat",
                "split_id",
                "num_layers",
                "agg_type",
                "llama_model",
                "desc_seg",
                "alpha",
                "stage",
                "feature",
            ]
        )
        .apply(get_summary)
        .reset_index()
    )
    return result

In [ ]:
parent_base = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/parent/coef_df_combined_new.parquet"
)
self_base = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/self/coef_df_combined_new.parquet"
)

In [ ]:
parent_coef_df = get_avg_coef_across_repeats(parent_base)
self_coef_df = get_avg_coef_across_repeats(self_base)

In [ ]:
coef_path_parent = coef_dir / "parent.parquet"
coef_path_self = coef_dir / "self.parquet"
if coef_path_parent.exists():
    coef_path_parent.unlink()
if coef_path_self.exists():
    coef_path_self.unlink()
coef_path_parent.parent.mkdir(exist_ok=True, parents=True)

parent_coef_df.to_parquet(coef_path_parent)
self_coef_df.to_parquet(coef_path_self)

## Examine tables

In [ ]:
parent_coef_df = pd.read_parquet(coef_dir / "parent.parquet")
self_coef_df = pd.read_parquet(coef_dir / "self.parquet")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


def get_heatmap(agg_type, llama, desc_seg, alpha, num_layers, df):
    sub_df = df[
        (df["agg_type"] == agg_type)
        & (df["llama_model"] == llama)
        & (df["desc_seg"] == desc_seg)
        & (df["alpha"] == alpha)
        & (df["num_layers"] == num_layers)
    ][["repeat", "stage", "feature", "avg_coef"]]
    sub_df["stage_feat"] = sub_df["stage"] + "_" + sub_df["feature"]
    ## PLOT
    # Pivot: rows = stage_feat, columns = repeat, values = avg_coef
    pivot = sub_df.pivot(index="stage_feat", columns="repeat", values="avg_coef")
    cmap = LinearSegmentedColormap.from_list(
        "blue_white_green",
        ["#08306b", "white", "#00441b"],  # dark blue, white, dark green
    )
    abs_max = pivot.abs().max().max()
    fig, ax = plt.subplots(figsize=(len(pivot.columns) * 0.8 + 1, 12))

    sns.heatmap(
        pivot,
        annot=True,  # show values in cells
        fmt=".2f",  # format to 2 decimal places
        cmap=cmap,
        center=0,
        vmin=-abs_max,
        vmax=abs_max,
        linewidths=0.5,
        ax=ax,
    )

    ax.set_title("avg_val by Repeat and Feature")
    ax.set_xlabel("Repeat")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

In [ ]:
impact_type = "parent"
agg_type = "proba-limited"
llama = "Llama-3.3-8B-Instruct"
desc_seg = "original"
alpha = "10.0_1.0"
num_layers = 3
df = self_base.copy() if impact_type == "self" else parent_base.copy()
###################################
###################################
df["repeat_split_id"] = df["repeat"].astype(str) + "_" + df["split_id"].astype(str)
# df.drop(['repeat', 'stage'], axis=1, inplace=True)
sub_df = df[
    (df["agg_type"] == agg_type)
    & (df["llama_model"] == llama)
    & (df["desc_seg"] == desc_seg)
    & (df["alpha"] == alpha)
    & (df["num_layers"] == num_layers)
][["repeat_split_id", "stage", "feature", "coef"]]
sub_df = sub_df.reset_index(drop=True)
# display(sub_df)
sub_df["stage_feat"] = sub_df["stage"] + "_" + sub_df["feature"]
# display(sub_df)
## PLOT
# Pivot: rows = stage_feat, columns = repeat, values = avg_coef
pivot = sub_df.pivot(index="stage_feat", columns="repeat_split_id", values="coef")


cmap = LinearSegmentedColormap.from_list(
    "blue_white_green", ["#08306b", "white", "#00441b"]  # dark blue, white, dark green
)
abs_max = pivot.abs().max().max()
fig, ax = plt.subplots(figsize=(len(pivot.columns) * 0.8 + 1, 12))

sns.heatmap(
    pivot,
    annot=True,  # show values in cells
    fmt=".2f",  # format to 2 decimal places
    cmap=cmap,
    center=0,
    # vmin=-abs_max,
    # vmax=abs_max,
    vmin=-3,
    vmax=3,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("avg_val by Repeat and Feature")
ax.set_xlabel("Repeat")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
sub_df.stage_feat.nunique()

In [ ]:
len(sub_df)

In [ ]:
sub_df.loc[sub_df.stage_feat.duplicated()]

In [ ]:
get_heatmap(
    agg_type="max",
    llama="Llama-3.3-8=70B-Instruct",
    desc_seg="pt_noshort",
    alpha="10.0",
    num_layers=2,
    df=parent_base,
)

# Experiment

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.residual_func import get_lr_output

In [ ]:
get_lr_output(
    df,
    agg_type="proba-limited",
    num_layers=2,
    selected_alphas=[0.5],
    selected_l1s=[None],
    split_id=2,
    impact_type="parent",
    impactme_outcome=None,
    max_iter=4000,
    seed=424242,
    linear_type="ridge",
    perm_dir=BASE_PATH / "dummy",
    hypothesis_shuffle=None,
)